# Notebook 2 -- How a single market evolves over time

Notebook 1 worked out what each column means and which ones matter. This notebook
picks up from that conclusion and asks a different question: **for one market, how
do the orderbook and the trade tape move together over time?**

It is deliberately self-contained -- it re-reads the parquet files and rebuilds the
derived columns, so notebook 1 does not have to be run first.

Key fact carried over from notebook 1: every books row -- `snapshot` **and** `update` --
contains a full top-5 ladder on both sides. So a row is a complete photo of the book,
not a delta. Nothing in the data labels *why* the book changed, so the cause
(someone added, someone cancelled, or someone traded) has to be inferred by comparing
consecutive photos and lining the trades up against them.

### Setup: libraries, raw data, derived columns

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import get_market, show_window

In [2]:
df_trades = pd.read_parquet("trades_823753_pregame.parquet")
df_books  = pd.read_parquet("orderbook_data_823753_pregame.parquet")

print("books ", df_books.shape)
print("trades", df_trades.shape)

books  (58275, 18)
trades (2232, 18)


In [3]:
# Convert the exchange and received timestamps from object/string to datetime, with time zone awareness
for df in [df_trades, df_books]:
    for col in ["recv_ts_utc", "exchange_ts_utc"]:
        df[col] = pd.to_datetime(df[col], utc = True)

In [4]:
# Add a column at the 1-second granularity, because millisecond and microsecond makes it hard to analyze the event
# Note: floor(), not round() -- floor gives clean bins that line up with resample/Grouper,
# whereas rounding shifts events across bin boundaries at the half-second.
for df in (df_books, df_trades):
    for ts in ["recv_ts_utc", "exchange_ts_utc"]:
        df[ts.replace("_utc", "_sec")] = df[ts].dt.floor("s")

In [5]:
# Latency between the exchange timestamp and the time we received the message
for df in (df_trades, df_books):
    df["latency_s"] = (df["recv_ts_utc"] - df["exchange_ts_utc"]).dt.total_seconds()

### Slice down to the columns that matter

Same verdict as notebook 1, with one deliberate difference: **the raw microsecond
timestamps come back.** Notebook 1 worked at 1-second granularity, which was right for
aggregate EDA. Here a single market can take dozens of updates inside one second, so
`_sec` would scramble the order of exactly the events we are trying to watch.

Both raw clocks are kept, but **sequencing is done on `recv_ts_utc` throughout**, for one
reason: `exchange_ts_utc` is quantized to whole milliseconds and so cannot separate events
that land in the same millisecond. Across all 33 markets, 6,606 of 58,275 book rows
(11.3%) share an exchange timestamp with another row; `recv_ts_utc` has zero duplicates.
The 757-contract trade below is the case in point -- the book update and the trade it
caused carry the *identical* exchange timestamp, so only the receive clock can tell us
which came first.

Worth recording what is *not* wrong with the exchange clock: sorted by `recv_ts_utc`,
`exchange_ts_utc` never runs backwards -- 0 inversions across all 58,275 rows, snapshots
included. Snapshots do restate stale times (median latency 6.6s vs 0.023s for updates),
which distorts the latency histogram, but it does not reorder events within a market.
So exchange time is the better clock for measuring *elapsed time* between events, since
it is free of network jitter; receive time is the better clock for *ordering* them.

In [7]:
# Dropped everywhere: river_id (1:1 with native_id), game_pk (single game),
#   player_id / player_name (all NaN), market_type (only ever "Over", and only
#   populated for team_totals), collector_run_id (constant), date / hour (derived
#   from recv_ts_utc).

# Identity of the contract. native_id is the ticker; (stat, line, outcome_name)
# is the same information in human-readable form -- verified 1:1 both ways in notebook 1.
cols_id = ["native_id", "stat", "line", "outcome_name"]

# recv_ts_utc leads: raw microseconds, the only clock with no ties, so it defines
# event order. exchange_ts_utc rides along for cross-checking elapsed time.
cols_time = ["recv_ts_utc", "exchange_ts_utc", "latency_s"]

cols_books = [
    *cols_time,
    "msg_type",                        # update vs snapshot -- must filter on this
    *cols_id,
    "best_bid", "best_ask", "spread",
    "bids", "asks",                    # full top-5 depth, needed for liquidity analysis
]

cols_trades = [
    *cols_time,
    *cols_id,
    "price", "qty",
    "aggressor_buy_flag",              # True = aggressor lifted the ask (bought)
                                       # False = aggressor hit the bid (sold)
    "exchange_trade_id",
]
# msg_type omitted from trades: constant "trade"

books  = df_books[cols_books].copy()
trades = df_trades[cols_trades].copy()
print("books ", books.shape)
print("trades", trades.shape)

books  (58275, 13)
trades (2232, 11)


### Pick a market

`get_market` slices both tables down to one `native_id`, returns copies sorted
oldest-first on the raw `recv_ts_utc`, and keeps each row's position in the full
frame as `orig_idx` so we can always jump back.

In [9]:
# How many trades did each market actually see? Pick one with enough activity to watch.
trades["native_id"].value_counts().head(n = 10)

KXMLBRFI-26AUG051940PITMIL               1234
KXMLBTOTAL-26AUG051940PITMIL-8            385
KXMLBF5TOTAL-26AUG051940PITMIL-4          107
KXMLBTOTAL-26AUG051940PITMIL-6             71
KXMLBTOTAL-26AUG051940PITMIL-7             71
KXMLBTEAMTOTAL-26AUG051940PITMIL-MIL4      70
KXMLBTOTAL-26AUG051940PITMIL-5             38
KXMLBTOTAL-26AUG051940PITMIL-3             38
KXMLBTOTAL-26AUG051940PITMIL-9             28
KXMLBF5TOTAL-26AUG051940PITMIL-5           24
Name: native_id, dtype: Int64

In [10]:
MARKET = "KXMLBTOTAL-26AUG051940PITMIL-9"      # P(total runs in the game >= 9)

b, t = get_market(MARKET, books, trades)
print(f"{MARKET}\n  book rows: {len(b):,}   ({b['msg_type'].value_counts().to_dict()})")
print(f"  trades   : {len(t):,}")

KXMLBTOTAL-26AUG051940PITMIL-9
  book rows: 4,370   ({'update': 4209, 'snapshot': 161})
  trades   : 28


In [12]:
# The trades, so we have timestamps to walk through one at a time.
t[["recv_ts_utc", "price", "qty", "aggressor_buy_flag"]].head(n = 10)

,recv_ts_utc,price,qty,aggressor_buy_flag
0,2026-08-05 17:40:37.420654+00:00,0.37,18.56,False
1,2026-08-05 18:05:50.351606+00:00,0.37,13.00,False
2,2026-08-05 18:27:59.421822+00:00,0.37,4.00,False
3,2026-08-05 18:27:59.421842+00:00,0.37,73.36,False
4,2026-08-05 18:47:11.770280+00:00,0.36,12.00,False
5,2026-08-05 20:00:57.547629+00:00,0.36,8.45,False
6,2026-08-05 20:03:30.795219+00:00,0.36,7.00,False
7,2026-08-05 20:30:19.542399+00:00,0.36,76.20,False
8,2026-08-05 21:19:07.295179+00:00,0.36,95.10,False
9,2026-08-05 21:31:17.521298+00:00,0.36,4.00,False


### Walk through the book one window at a time

`show_window(b, t, start, seconds=1.0)` prints every book row and every trade inside a
time window, interleaved in time order. Each book row is rendered as the "whiteboard"
-- asks on top, bids below -- with the change in size at each price level versus the
previous book row.

Useful arguments:

| argument | meaning |
| --- | --- |
| `seconds` | window length; widen it to scan a quiet stretch |
| `full=False` | one compact line per book row instead of the whole ladder |
| `depth` | levels shown per side (5 is all the feed gives us) |
| `max_events` | guard so a busy second doesn't print hundreds of lines |

In [13]:
# A trade of 757 contracts. Watch the size disappear from the bid, then the trade print.
show_window(b, t, "2026-08-05 22:57:56")

2026-08-05 22:57:56.000000 -> 22:57:57.000000   5 book rows, 1 trades
  22:57:56.174813  update    best 0.36 / 0.37   (row 51402)
      ASK  0.41 x    10,969.00  
      ASK  0.40 x    16,384.00  
      ASK  0.39 x    66,914.10  
      ASK  0.38 x    37,716.68  
      ASK  0.37 x     6,175.50  
      ----------------------------------  spread 0.01
      BID  0.36 x    42,774.00  <- -757.00
      BID  0.35 x    67,443.97  
      BID  0.34 x    54,653.00  
      BID  0.33 x     8,966.90  
      BID  0.32 x     9,515.00  

  22:57:56.175308  *** TRADE  SELL     757.00 @ 0.36 ***

  22:57:56.227335  update    best 0.36 / 0.37   (row 51403)
      ASK  0.41 x    10,969.00  
      ASK  0.40 x    16,384.00  
      ASK  0.39 x    66,914.10  
      ASK  0.38 x    47,671.68  <- +9,955.00
      ASK  0.37 x     6,175.50  
      ----------------------------------  spread 0.01
      BID  0.36 x    42,774.00  
      BID  0.35 x    67,443.97  
      BID  0.34 x    54,653.00  
      BID  0.33 x     8,966

In [14]:
# Same stretch, compact, over a wider window -- easier to scan for what changed.
show_window(b, t, "2026-08-05 22:57:50", seconds = 15, full = False)

2026-08-05 22:57:50.000000 -> 22:58:05.000000   7 book rows, 1 trades
  22:57:56.174813  update    best 0.36 / 0.37   (row 51402)   bid 0.36 -757.00
  22:57:56.175308  *** TRADE  SELL     757.00 @ 0.36 ***

  22:57:56.227335  update    best 0.36 / 0.37   (row 51403)   ask 0.38 +9,955.00
  22:57:56.265101  update    best 0.36 / 0.37   (row 51404)   bid 0.36 -1,054.97; bid 0.34 +2,000.00
  22:57:56.292906  update    best 0.36 / 0.37   (row 51405)   ask 0.39 -9,888.00
  22:57:56.363422  update    best 0.36 / 0.37   (row 51406)   ask 0.39 -2,000.00; ask 0.37 +2,000.00
  22:57:58.545477  update    best 0.36 / 0.37   (row 51410)   bid 0.34 +22.54
  22:58:00.757244  update    best 0.36 / 0.37   (row 51415)   bid 0.34 -22.54


### To do

- Step through all of this market's trades and check the disappearing size matches `qty` each time.
- Look at quiet stretches: what does the book do when nobody is trading?
- Repeat on a busy market (`KXMLBRFI-...` has 1,234 trades) and a thin one, and compare.